# Quantumograph / FLQC — Interactive Phase-Floor Demo

**What this notebook shows.** The FLQC (Finite Lattice Quantum Computation) hypothesis
predicts that spacetime discreteness imposes a minimum resolvable phase increment,
Δθ_min, on any physically realizable quantum gate. In the real theory this floor is
tiny — Δθ_min ~ 10⁻³⁰ rad — far too small to see directly on any near-term hardware
(see the manuscript, §4.3, for why).

This notebook does **not** attempt to reproduce that real, invisible number. Instead
it lets you dial Δθ up to a pedagogically visible size (10⁻¹–10⁻³) so you can *see*
the qualitative mechanism: a **deterministic, bounded phase-rounding error** produces
a different signature — in shape, not just size — than ordinary **random gate noise**
of the same magnitude.

Run every cell top to bottom, then play with the sliders at the end.

*Part of the Quantumograph v14 project. See the FLQC manuscript and
[FLQC-Protocol](https://github.com/SergejMaterov/FLQC-Protocol) for the real
(non-pedagogical) measurement protocol and its feasibility analysis.*


## 1. Setup

In Google Colab, this cell installs everything needed (takes ~30s the first time).

In [ ]:
# If running in Google Colab, uncomment the line below:
# !pip install -q qiskit qiskit-aer ipywidgets

import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, state_fidelity
from ipywidgets import interact, FloatLogSlider, IntSlider


## 2. Two circuit models

Both start from the standard Quantum Fourier Transform (QFT) circuit — the same
phase-heavy primitive used by Shor's algorithm and discussed in the FLQC manuscript,
Part V.

- **`build_qft_flqc`** — every controlled-phase angle is rounded to the nearest
  multiple of Δθ (a discrete phase lattice, as in the FLQC hypothesis). The error
  per gate is *deterministic* and *bounded* by Δθ/2.
- **`build_qft_noisy`** — every controlled-phase angle gets independent Gaussian
  noise of standard deviation σ (an ordinary engineering-noise model — miscalibration,
  crosstalk, etc). The error per gate is *random* and *unbounded*.

We compare their effect on circuit fidelity as a function of register size `n`.


In [ ]:
def _apply_input(qc, n, input_int):
    '''Prepare a fixed, non-symmetric computational-basis input so the
    controlled-phase gates are actually exercised (an all-zero input trivially
    hides any phase error).'''
    bits = format(input_int % (2 ** n), f'0{n}b')[::-1]
    for k, b in enumerate(bits):
        if b == '1':
            qc.x(k)

def build_qft_flqc(n, delta_theta, input_int):
    '''QFT with every controlled-phase angle rounded to the nearest
    multiple of delta_theta -- the FLQC discrete phase lattice.'''
    qc = QuantumCircuit(n)
    _apply_input(qc, n, input_int)
    for i in range(n):
        qc.h(i)
        for j in range(i + 1, n):
            angle = np.pi / (2 ** (j - i))
            if delta_theta is not None:
                angle = np.round(angle / delta_theta) * delta_theta
            qc.cp(angle, j, i)
    return qc

def build_qft_noisy(n, sigma, input_int, rng):
    '''QFT with independent Gaussian noise of std sigma on every
    controlled-phase angle -- an ordinary engineering-noise model.'''
    qc = QuantumCircuit(n)
    _apply_input(qc, n, input_int)
    for i in range(n):
        qc.h(i)
        for j in range(i + 1, n):
            angle = np.pi / (2 ** (j - i)) + rng.normal(0, sigma)
            qc.cp(angle, j, i)
    return qc

def golden_input(n):
    '''A fixed, non-symmetric input integer, chosen so the demo isn't
    accidentally exercising a special-case (e.g. all-zero) input.'''
    return int(0.6180339887 * (2 ** n))


In [ ]:
def infidelity_curve_flqc(n_values, delta_theta):
    out = []
    for n in n_values:
        k = golden_input(n)
        ideal = Statevector.from_instruction(build_qft_flqc(n, None, k))
        phys = Statevector.from_instruction(build_qft_flqc(n, delta_theta, k))
        out.append(1 - state_fidelity(ideal, phys))
    return np.array(out)

def infidelity_curve_noisy(n_values, sigma, n_realizations=8, seed=0):
    rng = np.random.default_rng(seed)
    curves = []
    for _ in range(n_realizations):
        out = []
        for n in n_values:
            k = golden_input(n)
            ideal = Statevector.from_instruction(build_qft_flqc(n, None, k))
            noisy = Statevector.from_instruction(build_qft_noisy(n, sigma, k, rng))
            out.append(1 - state_fidelity(ideal, noisy))
        curves.append(out)
    return np.mean(curves, axis=0), np.std(curves, axis=0)


## 3. Interactive comparison

Move the sliders. Look for the **shape**, not just the height, of the curves:

- The **FLQC (rounded)** curve should rise and then visibly **flatten** — once a
  gate's ideal angle is finer than Δθ, that gate simply stops contributing new
  error, so the curve saturates.
- The **engineering noise (random)** curve, at comparable magnitude, keeps
  **climbing** with n instead of flattening, because independent random errors
  keep accumulating regardless of the phase-lattice spacing.

This shape difference — saturating vs. growing — is the qualitative signature
described in the manuscript, §4.1: *"does the deviation grow with n, or does it
remain bounded?"*

Note: the register sizes here (n ≤ 16) are small enough to simulate instantly,
but too small to show clean asymptotic saturation for very small Δθ — try a
larger Δθ (10⁻¹–10⁻²) first to see the effect clearly, since real Δθ_min is far
smaller than anything simulable this way (see the disclaimer in cell 1).

In [ ]:
n_values = list(range(2, 17))

def plot_comparison(log_delta_theta=-1.0, log_sigma=-1.0, n_realizations=6):
    delta_theta = 10 ** log_delta_theta
    sigma = 10 ** log_sigma

    inf_flqc = infidelity_curve_flqc(n_values, delta_theta)
    inf_noisy_mean, inf_noisy_std = infidelity_curve_noisy(
        n_values, sigma, n_realizations=n_realizations
    )

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(n_values, inf_flqc, 'o-', color='#d62728', linewidth=2,
            label=f'FLQC-style rounding (Δθ={delta_theta:.0e})')
    ax.plot(n_values, inf_noisy_mean, 's-', color='#1f77b4', linewidth=2,
            label=f'Random gate noise (σ={sigma:.0e}, mean of {n_realizations} runs)')
    ax.fill_between(n_values, inf_noisy_mean - inf_noisy_std,
                     inf_noisy_mean + inf_noisy_std, color='#1f77b4', alpha=0.15)
    ax.set_xlabel('QFT register size n (qubits)')
    ax.set_ylabel('Infidelity  1 − F')
    ax.set_title('Deterministic phase-lattice rounding vs. random gate noise')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

interact(
    plot_comparison,
    log_delta_theta=FloatLogSlider(value=-1, base=10, min=-3, max=-0.5, step=0.25,
                                    description='Δθ'),
    log_sigma=FloatLogSlider(value=-1, base=10, min=-3, max=-0.5, step=0.25,
                              description='σ (noise)'),
    n_realizations=IntSlider(value=6, min=1, max=20, step=1,
                              description='noise runs'),
);


## 4. Where to go next

- **The real (non-pedagogical) protocol and its feasibility analysis** live in the
  main FLQC manuscript (§4.1–4.3) and in the
  [FLQC-Protocol](https://github.com/SergejMaterov/FLQC-Protocol) repository.
- **Classical structural predictions** (spectral dimension of the ℤ⁴ graph, random
  walk return probability, etc.) are in the `quantumograph` package, a good next
  notebook to explore if this one was interesting to you.
- This notebook is deliberately self-contained and dependency-light (just Qiskit +
  matplotlib + ipywidgets) so it runs in Google Colab with no local setup.

**Citation.** If you use this notebook in coursework or a thesis, please cite the
FLQC manuscript (Materov S., *Quantumograph v14*) — see the main repository README
for the full reference.
